In [26]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

DATA_DIR = Path("..") / "data"
LABEL_NAMES = {0: "Background", 1: "Basis", 2: "Discuss", 3: "Differ", 4: "Support"}

train = pd.read_csv(DATA_DIR / "data_v100_train.csv")
val = pd.read_csv(DATA_DIR / "data_v100_val.csv")
test_pub = pd.read_csv(DATA_DIR / "data_v100_test.csv")   # labeled, reliable held-out set
test_final = pd.read_csv(DATA_DIR / "test.csv")           # unlabeled, real leaderboard set

def marker_stats(df, name):
    n_cite = df["citation_context"].str.count("<CITE>")
    n_ref = df["citation_context"].str.count(r"\[REF\]")
    n_double_space = df["citation_context"].str.count(r"\s{2,}")
    print(f"{name:12s} n={len(df):4d}  <CITE>!=1: {(n_cite != 1).sum():3d}  "
          f"has [REF]: {(n_ref > 0).sum():3d}  has double-space: {(n_double_space > 0).sum():3d}")

for name, df in [("train", train), ("val", val), ("test_pub", test_pub), ("test_final", test_final)]:
    marker_stats(df, name)

print("\nDouble-space examples (train):")
print(train.loc[train["citation_context"].str.count(r"\s{2,}") > 0, "citation_context"].head(5).to_string(index=False))

train        n=1866  <CITE>!=1:   0  has [REF]:  18  has double-space: 257
val          n= 330  <CITE>!=1:   0  has [REF]:   3  has double-space:  41
test_pub     n= 550  <CITE>!=1:   0  has [REF]:   8  has double-space:  79
test_final   n= 326  <CITE>!=1:  12  has [REF]:   0  has double-space:  39

Double-space examples (train):
                                                  Mikroservis temelli uygulamalarda servis iletişimleri, REST API  , GraphQL <CITE> , ve gRPC   gibi günümüzün yaygın iletişim protokolleri ile sağlanabilmektedir.
                                                                                                                    Bulanık kümeleme yaklaşımları görüntülerin kümelenmesinde sıklıkla kullanılmaktadır  - <CITE> .
                                                                                                       Genel anlamda 3B iki tür veri kümesi vardır: iç mekân sahneleri  ,   ve dış mekân kentsel sahneleri <CITE> .
          İleriki araştırmalar, 

test_final (the real leaderboard set) has 12 rows where <CITE> doesn't appear exactly once, and zero rows with [REF] — a different annotation convention than train/val/test_pub. This could break the core assumption that <CITE> unambiguously marks the target citation. Need to see these 12 rows immediately, since this affects every downstream model (how do we know which citation to classify if there's 0 or >1 <CITE>?).

In [27]:
n_cite_final = test_final["citation_context"].str.count("<CITE>")
anomalies = test_final.loc[n_cite_final != 1, ["id", "citation_context"]].copy()
anomalies["n_cite"] = n_cite_final[n_cite_final != 1]
pd.set_option("display.max_colwidth", 300)
print(anomalies.to_string(index=False))

   id                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            citation_context  n_cite
29949                                                                                                                                                                                                                                                                                                                                                                                                         

real data quality issues in the leaderboard test set:

* One row (id=46186) has a missing/NaN citation_context — we must still produce a prediction for it in the submission (can't skip a row), so we need an explicit fallback (e.g. predict majority class).
* 11 rows have 2-4 <CITE> markers instead of exactly 1 — unlike train/val/test_pub, this test set doesn't mask co-citations consistently. Since our chosen strategy is Flat whole-sentence classification (not position-based), this is survivable — the model just sees multiple <CITE> tokens in context — but it's a real train/test annotation-convention mismatch worth flagging in the report as a limitation, not something to over-engineer around for only 11/326 (3.4%) rows.

In [28]:
print(test_final[test_final["id"] == 46186].to_string())

print("\nNaN counts per column, all files:")
for name, df in [("train", train), ("val", val), ("test_pub", test_pub), ("test_final", test_final)]:
    print(name, dict(df.isnull().sum()))

        id citation_context                                                                                       section
279  46186              NaN  Algorithms and performances increasing the encoding speed of rsa algorithm Extended Abstract

NaN counts per column, all files:
train {'id': np.int64(0), 'citation_context': np.int64(0), 'section': np.int64(0), 'citation_intent': np.int64(0)}
val {'id': np.int64(0), 'citation_context': np.int64(0), 'section': np.int64(0), 'citation_intent': np.int64(0)}
test_pub {'id': np.int64(0), 'citation_context': np.int64(0), 'section': np.int64(0), 'citation_intent': np.int64(0)}
test_final {'id': np.int64(0), 'citation_context': np.int64(1), 'section': np.int64(0)}


In [29]:
test_final_ext = pd.read_csv(DATA_DIR / "test_ext.csv")
print(test_final_ext[test_final_ext["id"] == 46186].to_string())

        id                                                                                                                                                                                                                                                                                                                                                                                                                       citation_context                                                                                       section
279  46186  In this study, asymmetric and symmetric encryption is divided into two of these encryption methods asymmetric encryption method is discussed. The RSA <CITE> algorithm, which is used as the most broadcast among asymmetric encryption algorithms, was investigated. This algorithm relies on the difficulty of factoring large numbers which has seriously affected its performance and so restricts its use in wider applications.  Algorithms and performances increasin

test_ext.csv recovered a full, valid citation sentence for the missing row — and it reveals this specific paper is written in English, not Turkish (an "Extended Abstract" on RSA encryption speed). This is useful: we don't need a blind majority-class fallback for this row, we can substitute real text from _ext for it. It also reminds us that BERTurk/Turkish-only tokenizers may handle this one row's English text somewhat differently — worth a footnote, not a blocker (it's 1/326).

This suggests a smarter general policy: use _ext as a targeted fallback, not the default. Let's check whether the 11 multi-<CITE> rows also have a cleaner single-<CITE> version in test_ext.csv — if so, we can recover clean text for those too instead of just tolerating the ambiguity.

In [30]:
anomaly_ids = anomalies["id"].dropna().astype(int).tolist()
ext_check = test_final_ext[test_final_ext["id"].isin(anomaly_ids)][["id", "citation_context"]]
ext_check["n_cite_ext"] = ext_check["citation_context"].str.count("<CITE>")
print(ext_check.to_string(index=False))

   id                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   

All 12 problematic rows have exactly one <CITE> in test_ext.csv, with co-citations properly masked as [REF]/bracketed numbers. So this is a base-extraction artifact isolated to test.csv (train/val/test_pub had zero such issues) — not a fundamental data problem. We can patch it precisely:

In [31]:
def fix_test_final(df, df_ext):
    df = df.copy()
    n_cite = df["citation_context"].str.count("<CITE>")
    bad_mask = df["citation_context"].isna() | (n_cite != 1)
    ext_lookup = df_ext.set_index("id")["citation_context"]
    df.loc[bad_mask, "citation_context"] = df.loc[bad_mask, "id"].map(ext_lookup)
    return df, bad_mask.sum()

test_final_fixed, n_fixed = fix_test_final(test_final, test_final_ext)
print(f"Patched {n_fixed} rows using test_ext.csv fallback")

remaining_bad = test_final_fixed["citation_context"].isna() | (test_final_fixed["citation_context"].str.count("<CITE>") != 1)
print("Remaining bad rows:", remaining_bad.sum())

Patched 12 rows using test_ext.csv fallback
Remaining bad rows: 0


In [32]:
def clean_text(series: pd.Series) -> pd.Series:
    return series.str.replace(r"\s{2,}", " ", regex=True).str.strip()

for df in [train, val, test_pub, test_final_fixed]:
    df["citation_context_clean"] = clean_text(df["citation_context"])

print(train[["citation_context", "citation_context_clean"]].head(3).to_string())

from sklearn.utils.class_weight import compute_class_weight

classes = np.array(sorted(LABEL_NAMES.keys()))
weights = compute_class_weight(class_weight="balanced", classes=classes, y=train["citation_intent"])
class_weight_map = dict(zip(classes, weights))
print("\nClass weights (balanced):")
for c, w in class_weight_map.items():
    print(f"  {LABEL_NAMES[c]:12s} (label {c}): weight={w:.3f}")

                                                                                                                                                                                                                                                                                                                                                                                 citation_context                                                                                                                                                                                                                                                                                                                                                                          citation_context_clean
0                                                                                                                                                                                                                                     

Interpretation: Whitespace cleanup worked correctly (multi-space collapsed, single spaces before commas/punctuation preserved). Class weights confirm the imbalance severity numerically — Differ gets ~10x the weight of Background, Discuss/Support ~4.5-5x, Basis ~1.9x. These will go straight into the loss function (CrossEntropyLoss(weight=...)) in 04_training.ipynb.

One more Turkish-specific quirk worth checking before finalizing the <CITE> handling strategy: since Turkish is agglutinative, suffixes often attach directly to <CITE> (e.g. <CITE>'yi, <CITE>'de, <CITE>'da) rather than being separated by a space. This affects whether we should add <CITE>/[REF] as atomic special tokens to the BERT tokenizer.

In [33]:
suffix_pattern = train["citation_context_clean"].str.extractall(r"<CITE>(['’]\w+)")
print("Rows with suffix directly attached to <CITE>:", suffix_pattern.index.get_level_values(0).nunique(), "/", len(train))
print(suffix_pattern[0].value_counts().head(15))

Rows with suffix directly attached to <CITE>: 11 / 1866
0
’de      2
’daki    1
’den     1
'in      1
’da      1
'de      1
'den     1
’dan     1
’nın     1
’ya      1
Name: count, dtype: int64


Interpretation: Only 11/1866 rows (0.6%) have a suffix glued directly to <CITE>, with mixed curly/straight apostrophes (’de, 'in, etc.) — negligible edge case. Adding <CITE> (and [REF]) as atomic special tokens to the BERT tokenizer is safe: any trailing suffix will just be tokenized normally afterward as its own subword(s), which is exactly how Turkish tokenizers already handle agglutination.

### Save Final Processed Data
Two fixes applied before saving: collapse redacted-citation double-spaces, and replace Turkish İ → i (Python's default `.lower()` mishandles `İ`, turning it into `i` + a combining dot; ASCII `I` is left alone since it's only used here for acronyms like `API`/`IoT`)

In [41]:
def clean_text(series: pd.Series) -> pd.Series:
    fixed = series.str.replace("İ", "i", regex=False)
    return fixed.str.replace(r"\s{2,}", " ", regex=True).str.strip()

for df in [train, val, test_pub, test_final_fixed]:
    df["citation_context"] = clean_text(df["citation_context"])

PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DIR.mkdir(exist_ok=True)

train[["id", "citation_context", "citation_intent"]].to_csv(PROCESSED_DIR / "train.csv", index=False)
val[["id", "citation_context", "citation_intent"]].to_csv(PROCESSED_DIR / "val.csv", index=False)
test_pub[["id", "citation_context", "citation_intent"]].to_csv(PROCESSED_DIR / "test_pub.csv", index=False)
test_final_fixed[["id", "citation_context"]].to_csv(PROCESSED_DIR / "test_final.csv", index=False)

import json
with open(PROCESSED_DIR / "class_weights.json", "w") as f:
    json.dump({int(k): float(v) for k, v in class_weight_map.items()}, f, indent=2)

print("Saved to:", PROCESSED_DIR.resolve())
print(sorted(p.name for p in PROCESSED_DIR.iterdir()))

Saved to: C:\Users\Arda\Desktop\Engineering\NLPhomework1\data\processed
['class_weights.json', 'test_final.csv', 'test_pub.csv', 'train.csv', 'val.csv']


## Data Validation before continuing with training

before trusting any baseline numbers, let's check the simplest, most important data-integrity question first — are there duplicate/near-identical sentences appearing in both train and val/test_pub? If so, our Macro F1 numbers so far could be artificially inflated (the model would just be memorizing a sentence it already saw in training).

In [35]:
all_text = pd.concat([
    train[["id", "citation_context"]].assign(split="train"),
    val[["id", "citation_context"]].assign(split="val"),
    test_pub[["id", "citation_context"]].assign(split="test_pub"),
])
dup_text = all_text[all_text.duplicated(subset="citation_context", keep=False)]
print("Rows sharing identical citation_context text across splits:", len(dup_text))
print(dup_text.sort_values("citation_context").head(10).to_string(index=False))

Rows sharing identical citation_context text across splits: 2
   id                                                                                                citation_context split
29761 Covid -19 salgın süreci, kriz yönetimi bakımından bu tespite verilebilecek başarılı örneklerden biridir <CITE>. train
29759 Covid -19 salgın süreci, kriz yönetimi bakımından bu tespite verilebilecek başarılı örneklerden biridir <CITE>. train


both duplicate rows are within train itself (same split), not shared across train/val/test_pub. So there's no cross-split leakage — our val/test_pub Macro F1 numbers are trustworthy. This looks like the same citation sentence got extracted twice by mistake (duplicate row within train), which is harmless for evaluation, just slightly redundant training data.

In [36]:
train_ext = pd.read_csv(DATA_DIR / "data_v100_train_ext.csv")
merged = train.merge(train_ext, on="id", suffixes=("_base", "_ext"))

print("Same id sets:", set(train.id) == set(train_ext.id))
print("Rows matched after merge:", len(merged), "out of", len(train))
print("Label mismatches base vs ext:", (merged["citation_intent_base"] != merged["citation_intent_ext"]).sum())

Same id sets: True
Rows matched after merge: 1866 out of 1866
Label mismatches base vs ext: 0


train and train_ext are perfectly aligned (same 1866 ids, zero label mismatches). Our earlier Flat-vs-Ext comparison was valid.

In [37]:
for name, df in [("train", train), ("val", val), ("test_pub", test_pub)]:
    print(f"{name}: unique citation_intent = {sorted(df['citation_intent'].unique())}")

train: unique citation_intent = [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
val: unique citation_intent = [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
test_pub: unique citation_intent = [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]


In [38]:
bracket_pattern = r"\[\d+\]|\(\w+.*?\d{4}\)"
for name, df in [("train", train), ("val", val), ("test_pub", test_pub)]:
    n = df["citation_context"].str.contains(bracket_pattern, regex=True).sum()
    print(f"{name}: rows with bracket/year-citation patterns: {n} / {len(df)}")

train: rows with bracket/year-citation patterns: 3 / 1866
val: rows with bracket/year-citation patterns: 0 / 330
test_pub: rows with bracket/year-citation patterns: 1 / 550


Negligible — only 3/1866 train, 0/330 val, 1/550 test_pub. Base files are already well-cleaned. Quick peek to close this out, then we're done validating and can move to baselines:

In [39]:
pattern_rows = train[train["citation_context"].str.contains(bracket_pattern, regex=True)]
print(pattern_rows[["id", "citation_context"]].to_string(index=False))

   id                                                                                                                                                                                                                                            citation_context
29583                                                      Avrupa Birliği Komisyonu'nun Kritik Bilgi Altyapısı Koruma (CIIP) eylem planında <CITE> ön görüldüğü üzere 2010 yılı içerisinde Siber Avrupa 2010 (Cyber Europe 2010) denemeleri gerçekleştirilmiştir.
32657                                                                                                  Bunun yanı sıra, Çizelge-1'de başarılan en yüksek doğrusal olmama değeri (16272), daha önce<CITE> çalışmalarında elde edilen sonuçtan (16268) daha iyidir.
29480 Bilişim Alanları Müfredatı (The Computing Curriculum 2020-CC2020) bilişim alanlarında lisans düzeyinde müfredatlar için geliştirilmiş yetkinlik bazlı bir rehberdir ve yazılım mühendisliği alanında da taslak halinde bir a

those 3 are false positives (plain numbers like (16272), (16268), or a real year "2010" mentioned in the sentence itself, not a citation year). So actual leakage is zero. Validation phase is done and our data is confirmed clean.